# OpenPlaque RCA Source CCTA Diagnostic — main baseline
This notebook identifies the likely original axial CCTA series in `Full_DICOM.zip`, verifies patient-space geometry, and displays the selected source volume. It does not use the curved RCA reformat for centerline extraction.

Use **Runtime → Run all**. Google Drive mounts first.

In [ ]:
# ALWAYS FIRST
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Fresh clean branch + dependencies before any third-party imports.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK numpy matplotlib pandas

import os, sys, time, shutil, zipfile
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
import SimpleITK as sitk
SRC=Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
print('Dependencies ready.')


In [ ]:
# Stage and extract the study locally.
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP=ROOT/'Full_DICOM.zip'
LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT=Path('/content/full_dicom_source_diag')
if not DRIVE_ZIP.exists(): raise FileNotFoundError(f'Missing {DRIVE_ZIP}')
t=time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying {DRIVE_ZIP.name} ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB) to local disk...',flush=True)
    shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f}s',flush=True)
else:
    print('Local ZIP already staged.')
if EXTRACT.exists(): shutil.rmtree(EXTRACT)
print('Extracting locally...',flush=True)
t=time.time()
with zipfile.ZipFile(LOCAL_ZIP) as z: z.extractall(EXTRACT)
print(f'Extraction finished in {time.time()-t:.1f}s',flush=True)


In [ ]:
# Inventory DICOM series and rank likely original axial CCTA sources.
groups=defaultdict(list); headers={}
print('Scanning DICOM headers locally...',flush=True)
t=time.time()
for root,_,files in os.walk(EXTRACT):
    for fn in files:
        p=os.path.join(root,fn)
        try:
            ds=pydicom.dcmread(p,stop_before_pixels=True,force=True)
            uid=str(ds.SeriesInstanceUID)
            groups[uid].append(p)
            if uid not in headers: headers[uid]=ds
        except Exception:
            pass
print(f'Header scan finished in {time.time()-t:.1f}s; {len(groups)} series found.',flush=True)

def orientation_normal(ds):
    try:
        iop=np.asarray(ds.ImageOrientationPatient,dtype=float)
        return np.cross(iop[:3],iop[3:])
    except Exception:
        return None

rows=[]
for uid,paths in groups.items():
    ds=headers[uid]
    desc=str(getattr(ds,'SeriesDescription',''))
    imagetype=' | '.join(str(x) for x in getattr(ds,'ImageType',[]))
    n=len(paths); r=int(getattr(ds,'Rows',0) or 0); c=int(getattr(ds,'Columns',0) or 0)
    try: thick=float(getattr(ds,'SliceThickness',np.nan) or np.nan)
    except Exception: thick=np.nan
    pix=getattr(ds,'PixelSpacing',None)
    try: pix0=float(pix[0]) if pix is not None else np.nan
    except Exception: pix0=np.nan
    norm=orientation_normal(ds); nz=abs(float(norm[2])) if norm is not None else np.nan
    score=0.0
    if n>=100: score+=5
    if n>=200: score+=2
    if r==c and r>=256: score+=2
    if 'ORIGINAL' in imagetype.upper(): score+=2
    if np.isfinite(nz) and nz>0.85: score+=3
    if np.isfinite(thick) and thick<=1.5: score+=2
    dl=desc.lower()
    if any(k in dl for k in ['coronary','ccta','cardiac','arterial','cta']): score+=2
    if any(k in dl for k in ['curved','cpr','mpr','vessel','rca','lad','lcx','cx/','straightened','vr','3d']): score-=8
    rows.append(dict(uid=uid,series=int(getattr(ds,'SeriesNumber',-1)),description=desc,images=n,rows=r,cols=c,pixel_mm=pix0,slice_mm=thick,normal_z=nz,image_type=imagetype,score=score))
df=pd.DataFrame(rows).sort_values(['score','images'],ascending=[False,False]).reset_index(drop=True)
display(df.head(20)[['series','description','images','rows','cols','pixel_mm','slice_mm','normal_z','image_type','score']])

candidate_rows=df.head(min(6,len(df))).copy(); geometry=[]; ordered_files={}
for _,row in candidate_rows.iterrows():
    uid=row.uid; paths=groups[uid]; ds0=headers[uid]; norm=orientation_normal(ds0); vals=[]
    for p in paths:
        try:
            ds=pydicom.dcmread(p,stop_before_pixels=True,force=True)
            if norm is not None and hasattr(ds,'ImagePositionPatient'):
                pos=float(np.dot(np.asarray(ds.ImagePositionPatient,dtype=float),norm))
            else:
                pos=float(getattr(ds,'InstanceNumber',0))
            vals.append((pos,p))
        except Exception:
            pass
    vals.sort(key=lambda x:x[0]); ordered_files[uid]=[p for _,p in vals]
    pos=np.asarray([v for v,_ in vals],dtype=float); step=np.abs(np.diff(pos)) if len(pos)>1 else np.array([])
    med=float(np.median(step)) if len(step) else np.nan
    cv=float(np.std(step)/med) if len(step) and med>0 else np.nan
    span=float(abs(pos[-1]-pos[0])) if len(pos)>1 else np.nan
    geometry.append(dict(series=int(row.series),description=row.description,images=len(vals),median_step_mm=med,step_cv=cv,span_mm=span,score=float(row.score)))
gdf=pd.DataFrame(geometry)
display(gdf)
good=gdf[(gdf.images>=100)&(gdf.median_step_mm>0)&(gdf.median_step_mm<=2.5)&(gdf.step_cv<0.20)]
chosen=good.iloc[0] if len(good) else gdf.iloc[0]
SOURCE_SERIES=int(chosen.series)
source_uid=df.loc[df.series==SOURCE_SERIES,'uid'].iloc[0]
print('Automatically selected source candidate series:',SOURCE_SERIES,chosen.description)


In [ ]:
# Load selected source volume and show conventional orthogonal views.
files_sel=ordered_files[source_uid]
if len(files_sel)<2: raise ValueError('Selected series has too few ordered slices.')
reader=sitk.ImageSeriesReader(); reader.SetFileNames(files_sel)
source_img=reader.Execute(); source=sitk.GetArrayFromImage(source_img)
print('Source series:',SOURCE_SERIES)
print('Shape zyx:',source.shape)
print('Spacing xyz mm:',source_img.GetSpacing())
print('Origin xyz:',source_img.GetOrigin())
print('Direction:',source_img.GetDirection())
print('HU min/median/max:',float(np.min(source)),float(np.median(source)),float(np.max(source)))
z=source.shape[0]//2; y=source.shape[1]//2; x=source.shape[2]//2
fig,axes=plt.subplots(1,3,figsize=(16,5))
axes[0].imshow(source[z],cmap='gray',vmin=-200,vmax=800); axes[0].set_title(f'Axial z={z}')
axes[1].imshow(source[:,y,:],cmap='gray',vmin=-200,vmax=800,aspect='auto'); axes[1].set_title(f'Coronal y={y}')
axes[2].imshow(source[:,:,x],cmap='gray',vmin=-200,vmax=800,aspect='auto'); axes[2].set_title(f'Sagittal x={x}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
zs=np.linspace(max(0,int(source.shape[0]*.25)),min(source.shape[0]-1,int(source.shape[0]*.75)),9).round().astype(int)
fig,axes=plt.subplots(3,3,figsize=(12,12))
for ax,zz in zip(axes.ravel(),zs):
    ax.imshow(source[zz],cmap='gray',vmin=-200,vmax=800); ax.set_title(f'z={zz}'); ax.axis('off')
plt.tight_layout(); plt.show()
print('SOURCE CCTA DIAGNOSTIC COMPLETE.')
print('Please capture the ranked-series table, geometry table, and source-volume images.')
